# Обучение распознавателя архитектуры TRBA

## 0. Введение

Этот ноутбук показывает, как обучить собственный распознаватель TRBA с помощью **manuscript-ocr версии 0.1.13**, а затем использовать обученную модель для распознавания текста.

Для примера взят [DonkeySmallOCR-Numbers-Printed-15random](https://huggingface.co/datasets/sherstpasha/DonkeySmallOCR-Numbers-Printed-15random). Это небольшой набор синтетических изображений с печатными числами и словами на кириллице. В нём 13 500 обучающих и 1 500 валидационных изображений. Архивы занимают около 30 МБ и распаковываются средствами Python.

Мы загрузим данные, посмотрим один пример, обучим лёгкий вариант TRBA и проверим экспортированную ONNX-модель на изображении из валидационной выборки.

Для обучения рекомендуется GPU. В Google Colab его можно включить в настройках среды выполнения. Если GPU недоступен, пример запустится на CPU, но обучение займёт больше времени.

В примере задана одна эпоха: этого достаточно для проверки запуска, но не для получения качественной модели. Количество эпох можно изменить в разделе с настройками.

## 1. Установка зависимостей

Устанавливаем библиотеку с зависимостями для обучения и клиент Hugging Face для загрузки датасета. Если после установки среда попросит перезапустить ядро, сделайте это перед запуском следующих ячеек.

В Colab обычно уже установлен PyTorch с поддержкой CUDA; переустанавливать его отдельно не требуется.

In [1]:
%pip install "manuscript-ocr[dev]==0.1.13" "huggingface-hub>=0.20"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 179.7 kB/s eta 0:00:001m190.8 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 281.7 kB/s eta 0:00:00 kB/s eta 0:00:01:01

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Настройки примера

Здесь можно изменить папки данных и эксперимента, количество эпох (`EPOCHS`) и размер пакета (`BATCH_SIZE`). Устройство выбирается автоматически: CUDA при наличии совместимого GPU, иначе CPU. При нехватке видеопамяти уменьшите `BATCH_SIZE`.

In [2]:
from pathlib import Path
import csv
from zipfile import ZipFile

import torch
from PIL import Image
from IPython.display import display
from huggingface_hub import snapshot_download
from manuscript.recognizers import TRBA
from manuscript.utils import create_page_from_image

DATASET = "sherstpasha/DonkeySmallOCR-Numbers-Printed-15random"
DATA_DIR = Path("datasets/donkey_small_ocr")
EXP_DIR = Path("experiments/trba_printed")
EPOCHS = 1
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство:", DEVICE)

/Users/user/manuscript-ocr/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Устройство: cpu


## 3. Загрузка и подготовка данных

Следующая ячейка скачает обучающую и валидационную выборки в папку `DATA_DIR`. Для повторного запуска уже скачанные файлы используются из локального кеша.

Каждая выборка содержит изображения в папке `img` и CSV с двумя столбцами: `image` — имя файла, `text` — правильный ответ. Обучающие данные используются для обновления весов, валидационные — для проверки качества после эпохи.

Алфавит модели формируется автоматически из символов обучающих подписей и сохраняется в `charset.txt`. После загрузки вы увидите количество примеров в каждой выборке.

In [3]:
snapshot_download(
    repo_id=DATASET,
    repo_type="dataset",
    local_dir=DATA_DIR,
    allow_patterns=["train.zip", "val.zip", "train.csv", "val.csv"],
)
for split in ("train", "val"):
    if not (DATA_DIR / split / "img").is_dir():
        with ZipFile(DATA_DIR / f"{split}.zip") as archive:
            archive.extractall(DATA_DIR)

samples = {}
for split in ("train", "val"):
    with (DATA_DIR / f"{split}.csv").open(encoding="utf-8", newline="") as stream:
        samples[split] = list(csv.DictReader(stream))
    print(split, len(samples[split]))

# Собственный алфавит: служебные токены и символы обучающих подписей.
characters = sorted({char for row in samples["train"] for char in row["text"]})
charset_path = DATA_DIR / "charset.txt"
charset_path.write_text("\n".join(["<PAD>", "<SOS>", "<EOS>", "<BLANK>", *characters]) + "\n", encoding="utf-8")

Fetching 4 files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:11<00:00,  2.92s/it]


train 13500
val 1500


190

### Посмотрим пример

Покажем первый валидационный кроп и его правильную подпись. На этом изображении затем проверим обученную модель.

In [4]:
sample = samples["val"][0]
image_path = DATA_DIR / "val" / "img" / sample["image"]
display(Image.open(image_path))
print("Эталон:", sample["text"])

Эталон: 6997549830


## 4. Запуск обучения

Обучение запускается через `TRBA.train()`. В этом примере используется лёгкая архитектура `seresnet31lite`; веса инициализируются с нуля, предобученная модель не загружается.

Во время обучения выводятся ошибки на train и val, точность распознавания и метрики CER/WER. Результаты сохраняются в папку `EXP_DIR`.

После завершения библиотека автоматически экспортирует лучшую по валидационной точности модель. Для последующего распознавания нужны три файла:

- `best_acc_model.onnx` — веса модели;
- `config.json` — настройки архитектуры и уровня распознавания;
- `charset.txt` — алфавит.

Сообщения `[OK] ONNX model is valid` и `[OK] Outputs match!` означают, что модель прошла проверку экспорта. Строка `[OK] Dynamic batch inference works for batch_size=2` подтверждает работу пакетного инференса.

Упрощение графа через `onnxsim` по умолчанию отключено в `TRBA.train()`: в некоторых окружениях оно может вызвать падение всего процесса. Сам экспорт и проверка ONNX выполняются. При необходимости упрощение можно включить параметром `export_simplify=True`.

In [5]:
training_result = TRBA.train(
    train_csvs=str(DATA_DIR / "train.csv"),
    train_roots=str(DATA_DIR / "train" / "img"),
    val_csvs=str(DATA_DIR / "val.csv"),
    val_roots=str(DATA_DIR / "val" / "img"),
    exp_dir=str(EXP_DIR),
    charset_path=str(charset_path),  # Алфавит задаёт допустимые символы и их индексы.
    cnn_backbone="seresnet31lite",  # Лёгкий CNN для извлечения признаков изображения.
    pretrain_weights=None,  # Обучаем с нуля; готовые веса не скачиваются.
    # Сохраняется в конфиге: при инференсе распознавать элементы Page типа word.
    # Это выбор областей, а не разбиение изображения на слова при обучении.
    recognition_levels=["word"],
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    val_size=len(samples["val"]),  # Оценивать всю валидационную выборку.
    num_workers=0,  # Загрузка данных в основном процессе: удобно для ноутбуков.
    device=DEVICE,
)
print(training_result)

[2026-10-13 20:03:50] Start training
[2026-10-13 20:03:50] Experiment dir: experiments/trba_printed
[2026-10-13 20:03:50] Seed: 42
[2026-10-13 20:03:50] Saved config to exp_dir/config.json
[2026-10-13 20:03:50] Device: cpu
[2026-10-13 20:03:50] Charset loaded: 86 tokens
[2026-10-13 20:03:50] Charset copied to experiment dir: experiments/trba_printed/charset.txt
[2026-10-13 20:03:50] Freeze policy applied: cnn: NONE (no freezing)
[2026-10-13 20:03:50] Freeze policy applied: enc_rnn: NONE (no freezing)
[2026-10-13 20:03:50] Freeze policy applied: attention: NONE (no freezing)
[2026-10-13 20:03:50] Parameters: trainable=10,452,588 | frozen=0 | total=10,452,588
[2026-10-13 20:03:50] Optimizer: AdamW, foreach=False
/Users/user/manuscript-ocr/src/manuscript/recognizers/_trba/training/train.py:1105: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()
/Users/user/manuscript-ocr/env/lib/python

[OCRDatasetAttn] Lazy image validation is enabled; unreadable images will be skipped during the first access.


[2026-10-13 20:03:50] Dataset sizes:                                                                                                                
[2026-10-13 20:03:50] ──────────────────────────────────────────────────────────────────
[2026-10-13 20:03:50]  set  train_csv                                    train       val
[2026-10-13 20:03:50] ──────────────────────────────────────────────────────────────────
[2026-10-13 20:03:50]    0  train                                        13500      1500
[2026-10-13 20:03:50] ──────────────────────────────────────────────────────────────────
[2026-10-13 20:03:50] TOTAL                                               13500      1500
[2026-10-13 20:03:50] ──────────────────────────────────────────────────────────────────
[2026-10-13 20:03:50]   set_0: separate val set: datasets/donkey_small_ocr/val/img
[2026-10-13 20:03:50] Loaders: train_batches/epoch=422; val_batches=47; batch_size=32
[2026-10-13 20:03:50] OneCycleLR: max_lr=0.002000, total_s

[OCRDatasetAttn] Lazy image validation is enabled; unreadable images will be skipped during the first access.


[2026-10-13 20:03:51] Logged augmentation previews to TensorBoard: 19 augmentations
[2026-10-13 20:03:51] Running epoch-0 baseline validation (before training)...
[2026-10-13 20:03:51] ─────────────────────────────────────────────────────
[2026-10-13 20:03:51]   set       n      loss       acc       CER       WER
[2026-10-13 20:03:51] ─────────────────────────────────────────────────────
/Users/user/manuscript-ocr/env/lib/python3.11/site-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
[2026-10-13 20:06:45]     0    1500    7.4623    0.0000    2.5647    1.0000
[2026-10-13 20:06:45] ─────────────────────────────────────────────────────
[2026-10-13 20:06:45] TOTAL    1500    7.4623    0.0000    2.5647    1.0000
[2026-10-13 20:06:45] ─────────────────────────────────────────────────────
                                                                                                             

KeyboardInterrupt: 

### Посмотрим созданные файлы

В папке эксперимента находятся и файлы для запуска модели, и материалы обучения.
Для распознавания достаточно ONNX, конфига и алфавита. Файлы `*_weights.pth`
содержат веса PyTorch, а `*_ckpt.pth` — состояние обучения для продолжения.
`best_acc` соответствует лучшей точности на val, `best_loss` — минимальной ошибке,
`last` — последней завершённой эпохе. Ниже выведем реальные файлы из папки, их размер и назначение.

In [ ]:
import os

EXP_DIR = Path(training_result["exp_dir"])
descriptions = {
    "best_acc_model.onnx": "Модель для инференса",
    "config.json": "Архитектура, параметры обучения и уровни распознавания",
    "charset.txt": "Алфавит и соответствие символов индексам",
    "metrics_epoch.csv": "Метрики обучения по эпохам",
    "logs": "Журнал обучения и события TensorBoard",
}
for name in sorted(os.listdir(EXP_DIR)):
    path = EXP_DIR / name
    description = descriptions.get(name)
    if description is None:
        if name.endswith("_ckpt.pth"):
            description = "Чекпоинт для продолжения обучения"
        elif name.endswith("_weights.pth"):
            description = "Веса модели PyTorch"
        else:
            description = "Дополнительный файл обучения"
    size = "папка" if path.is_dir() else f"{os.path.getsize(path) / (1024 ** 2):.2f} МБ"
    print(f"{name:26} {size:>10} — {description}")

## 5. Распознавание обученной моделью

Теперь загрузим экспортированную модель и распознаем тот же кроп, который посмотрели перед обучением. Изображение взято из валидационной выборки и не использовалось для обновления весов.

Передаём пути к весам, конфигу и алфавиту явно. `create_page_from_image()` создаёт область слова на весь кроп, а `recognizer.predict()` записывает распознанный текст и confidence в результат.

В выводе можно сравнить эталон с предсказанием модели. После одной эпохи ошибки ожидаемы: здесь мы проверяем весь путь от данных до инференса.

In [ ]:
# Для локальной TRBA нужны все три файла из одного эксперимента.
recognizer = TRBA(
    weights=str(EXP_DIR / "best_acc_model.onnx"),  # Обязательные ONNX-веса.
    config=str(EXP_DIR / "config.json"),          # Обязательные настройки модели и уровней распознавания.
    charset=str(EXP_DIR / "charset.txt"),        # Обязательный алфавит в порядке, использованном при обучении.
    device="cpu",
)
page = create_page_from_image(image_path, region_type="word")
result = recognizer.predict(page, image=image_path)
word = next(node for node in result.descendants if node.type == "word")
print("Эталон:", sample["text"])
print("Результат:", result.get_text(format="plain"))
print("Confidence:", word.recognition_confidence)